In [1]:
import sys
import os
os.chdir('/zhome/71/c/146676/main/')
import time
import SimpleITK as sitk
import numpy as np
from astropy.io import fits
from helpers import module_auxiliary as ma
import tifffile
from multiprocessing import Pool
import matplotlib.pyplot as plt
from helpers import extended_data as ed
from cil.recon import FBP
#from cil.plugins.astra import FBP
from cil.framework import AcquisitionGeometry, AcquisitionData, ImageGeometry, ImageData, BlockDataContainer
from cil.plugins.ccpi_regularisation.functions import FGP_TV
from cil.optimisation.functions import L2NormSquared, L1Norm, BlockFunction, MixedL21Norm, IndicatorBox, TotalVariation, LeastSquares
from cil.optimisation.operators import BlockOperator, GradientOperator, IdentityOperator, FiniteDifferenceOperator
from cil.optimisation.algorithms import CGLS, SIRT, GD, FISTA, ISTA, PDHG, SPDHG
from cil.plugins.astra.operators import ProjectionOperator
from cil.optimisation.functions import IndicatorBox, MixedL21Norm, L2NormSquared, \
                                       BlockFunction, L1Norm, LeastSquares, \
                                       OperatorCompositionFunction, TotalVariation, \
                                       ZeroFunction
from cil.optimisation.operators import BlockOperator, GradientOperator,\
                                       GradientOperator
import h5py
from cil.utilities.display import show_geometry
from scipy.ndimage import uniform_filter1d
from cil.processors import Slicer
from scipy.interpolate import interp1d
from loaders import stitcher_XA
from loaders import loader_XA_to_NA
import SimpleITK as sitk

In [2]:
c0 = 0
c1 = 2000

y0 = 120
y1 = 240

pad = 30

In [3]:
def zero_pad_3d(array: np.ndarray, N: int,value=0) -> np.ndarray:
    """
    Pads a 3D NumPy array with zeros along the last two dimensions by N pixels.
    The first dimension (axis=0) remains unchanged.

    Parameters:
    array (np.ndarray): Input 3D array.
    N (int): Number of pixels to pad along the last two dimensions.

    Returns:
    np.ndarray: Zero-padded 3D array.
    """
    if N < 0:
        raise ValueError("Padding size N must be non-negative")
    
    pad_width = ((0, 0), (0, 0), (N, N))  # No padding on axis=0, N on other axes
    return np.pad(array, pad_width, mode='constant', constant_values=0)


In [4]:
dataset = np.load('/dtu-compute/msaca/sliceA_diffraction/xrd_non_integrated/filtered_integrations/median_clipped_0_6.npy')
dataset = np.transpose(dataset/200,[2,0,1])[:]


with h5py.File('/dtu-compute/msaca/sliceA_diffraction/xrd_attenuation/scan-0339_xspress3-dtc-2d.h5', 'r') as file:
    # Check the keys in the file (this shows the main datasets or groups)
    
    # Access a dataset or group (replace 'your_dataset' with the correct key)
    dtc= file['entry/instrument/xspress3']
    data_all_events = dtc['all_events'][:].reshape(181,362)
    data_output_count_rate = dtc['output_count_rate'][:].reshape(181,362)
    data_att = dtc['window_counts'][:][:,0].reshape(181,362)

In [5]:
data = (1e3*dataset/data_att).astype(np.float32)
data_= zero_pad_3d(data, pad,value=np.mean(data[:,-10:]))
nchannels,ntheta, nx = np.shape(data)
nchannels,ntheta, nx_ = np.shape(data_)
recon_nolog = np.zeros((nchannels, y1-y0, nx))
angles = range(0,ntheta)

ag = AcquisitionGeometry.create_Parallel2D(detector_position=[0,nx_//2])\
                            .set_angles(angles)\
                            .set_channels(nchannels)\
                            .set_panel((nx_), pixel_size=(1))\
                            .set_labels(['channel','angle', 'horizontal'])
data_ = AcquisitionData(data_, geometry=ag)
data_.reorder('astra')
ig_ = ag.get_ImageGeometry(resolution=2)
roi = {'horizontal_x':(pad,-pad,1), 'horizontal_y':(pad+y0,pad+y1,1)}
processor = Slicer(roi)
processor.set_input(ig_)
ig = processor.get_output()
device = 'gpu'


In [6]:
xray_slices = range(0,2800) # Which xray slices to load? Starting from around 0 to around 3200
neutron_slices = range(0,1650) # Which neutron slices to load? Starting from 0 to around 1700. Slice 0 are at different ends of the meteorite
output_volume = [[0,1650],[None], [None]] # The slices included in the arrays that the registerred volumes are saved to (just set first coordinate 
# equal the numbers in Neutron slices, and the others to None (indicating to include all the data))


# Now read the datafrom disc, and register XA to NA. h=1 indicates that the neutron resolution should be used. Setting h higher, you can take advantage
# of the higher resolution of the xray data, but this requires changing output_volume
reg = loader_XA_to_NA.load_subset_of_registered_data(dataset_XA='tv', dataset_NA='tv' , h=1, xray_slices = xray_slices,
    neutron_slices = neutron_slices, output_volume = output_volume)
print('Loaded reconstructions')


Loading xray data
Loading neutron data
Resampling the transformation
Resampling moving image
Loaded reconstructions


In [ ]:

XA = sitk.GetArrayFromImage(reg.moving)

XA_d = XA[600-output_volume[0][0]:1000-output_volume[0][0]]



_ , nx_XA = np.shape(XA_d[0])
_ , nx_NA = np.shape(NA_d[0])
_ , nx_DA_2 = np.shape(data_att[0])

data_att_ = sitk.GetImageFromArray(data_att)
XA_ = sitk.GetImageFromArray(XA_d)
data_att_.SetSpacing((1/nx_DA_2,1/nx_DA_2,1/nx_DA_2))
XA_.SetSpacing((1/nx_XA,1/nx_XA,1/nx_XA))

size_fixed = data_att_.GetSize()
size_moving = XA_.GetSize()


spacing_fixed = data_att_.GetSpacing()
spacing_moving = XA_.GetSpacing()

# Compute the new origin (shift it to -N/2)
new_origin_fixed = [-0.5 * (size_fixed[i] - 1) * spacing_fixed[i] for i in range(len(size_fixed))]
new_origin_moving = [-0.5 * (size_moving[i] - 1) * spacing_moving[i] for i in range(len(size_moving))]

data_att_.SetOrigin(new_origin_fixed)
XA_.SetOrigin(new_origin_moving)


def resampler2(fixed, moving, transform, h=1):
    high_res=moving
    low_res=fixed
    # Get the spacing and size for the high-resolution image
    high_res_spacing = high_res.GetSpacing()
    high_res_size = high_res.GetSize()
    low_res_spacing = low_res.GetSpacing()
    low_res_size = low_res.GetSize()

    # Define the scaling factor
    scaling_factor = h  # Halve the spacing, double the size

    # Create the resampling filter
    resampler = sitk.ResampleImageFilter()

    # Set the output size (double the size)
    new_size = [int(dim * scaling_factor) for dim in low_res_size]

    # Set the output spacing (half the spacing)
    new_spacing = [spacing / scaling_factor for spacing in low_res_spacing]

    # Configure the resampler
    resampler.SetReferenceImage(low_res)  # Use the high-res image as reference
    resampler.SetSize(new_size)  # Set the new size
    resampler.SetOutputSpacing(new_spacing)  # Correct way to set spacing
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetTransform(transform)  # Apply the transformation
    resampler.SetOutputPixelType(fixed.GetPixelID())
    resampler.SetOutputOrigin(fixed.GetOrigin())  # Preserve origin
    resampler.SetOutputDirection(fixed.GetDirection())

    # Perform resampling
    return resampler.Execute(high_res)


h = 2
XA_registered_ = resampler2(data_att_, XA_, transform, h=h)
XA_registered = sitk.GetArrayFromImage(XA_registered_)

To-do:

Do the edge correction on xray data. (Do this in n12 notebook)


Check that XA is registered to existing volumes (just the data_att) [::h, ::h, ::h].

Save xray reconstruction for h=8.
(h=6?)

Do a slice reconstruction of original resolution, and check that the data is registerred. If not, change the image geometry....

Do reconstruction at h=2 resolution, and check the data is registerred.

Load h=8 xray image, apply a mean filter to the correct slice.
Decrease the size to h=2
Do dTV with this as reference image.
Increase h.